# Water Potability - Preprocessing and Model Analysis

This notebook documents the end-to-end preprocessing, leak-free train-validation-test splitting, multivariate iterative missing-value imputation, domain interaction feature engineering (`ph / Sulfate`), feature scaling, class imbalance handling, model comparison, and final evaluation.

In [1]:
import pandas as pd

df = pd.read_csv("../data/water_potability.csv")

print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Potability"].value_counts())

Dataset Shape: (3276, 10)

Missing Values:
ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

Target Distribution:
Potability
0    1998
1    1278
Name: count, dtype: int64


### Analysis of Schema, Missing Values, and Class Balance

- **Dataset Dimensions:** The dataset contains **3,276 water samples** across **9 physicochemical input features** and **1 binary target (`Potability`)**, with zero duplicate rows and no missing target labels.
- **Missingness Profile:** Three features have missing entries: `Sulfate` (**781 rows**, 23.84%), `ph` (**491 rows**, 14.99%), and `Trihalomethanes` (**162 rows**, 4.95%). Dropping rows with missing values would discard over 38% of the dataset, so imputation is preferred to retain full sample diversity.
- **Target Imbalance:** Non-potable water (`0`) accounts for **60.99%** (1,998 samples) while potable water (`1`) accounts for **39.01%** (1,278 samples), requiring stratified splitting and balanced class weights.

In [2]:
from sklearn.model_selection import train_test_split

FEATURES = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

X = df[FEATURES]
y = df["Potability"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (2293, 9)
Validation: (491, 9)
Testing: (492, 9)


### Analysis of Stratified Data Splitting

- The dataset is partitioned **before** any imputation, feature engineering, or scaling into **70% Training (2,293 rows)**, **15% Validation (491 rows)**, and **15% Testing (492 rows)**.
- Stratification guarantees identical class ratios (~61% Non-Potable / ~39% Potable) across all three splits and prevents data leakage from validation or test distributions into preprocessing statistics.

In [3]:
import joblib

preprocessor = joblib.load("../models/preprocessor.joblib")

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training:", X_train_processed.shape)
print("Processed Validation:", X_val_processed.shape)
print("Processed Testing:", X_test_processed.shape)

Processed Training: (2293, 10)
Processed Validation: (491, 10)
Processed Testing: (492, 10)


### Analysis of Multivariate Imputation, Domain Feature Engineering, and Scaling

- **Multivariate Iterative Imputation (`IterativeImputer` + `ExtraTreesRegressor`):** Rather than replacing 23.8% of `Sulfate` and 15.0% of `ph` values with a single global median (which artificially places non-potable records right at the neutral safe-water center), the pipeline predicts missing entries from the remaining physicochemical features.
- **Domain Feature (`ph_sulfate_ratio = ph / Sulfate`):** Appends the non-linear acid-to-sulfate balance ratio as the 10th feature (`shape = (_, 10)`), allowing decision trees to partition joint pH–Sulfate regimes directly.
- **Standardization (`StandardScaler`):** Normalizes all 10 features using training-set mean and standard deviation.

In [4]:
print("Training Class Distribution:")
print(y_train.value_counts())

print("\nTraining Class Proportions:")
print(y_train.value_counts(normalize=True))

Training Class Distribution:
Potability
0    1398
1     895
Name: count, dtype: int64

Training Class Proportions:
Potability
0    0.609682
1    0.390318
Name: proportion, dtype: float64


### Analysis of Class Imbalance Strategy

- Because the positive `Potable` class represents **39.03%** of the training split, unweighted classifiers bias toward predicting `Non-Potable` (`0`).
- Applying `class_weight="balanced"` inversely weights each class during split criterion evaluation and probability estimation, substantially improving minority-class recall and F1-score.

In [5]:
results = pd.read_csv("../outputs/validation_results.csv")
results.sort_values("ROC-AUC", ascending=False)

                                  Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
0  Proposed Model - Tuned Random Forest  0.668024   0.587500  0.492147  0.535613  0.672810
1                         Random Forest  0.670061   0.597315  0.465969  0.523529  0.667705
2                                   KNN  0.649695   0.588785  0.329843  0.422819  0.666195
3                                   SVM  0.635438   0.531579  0.528796  0.530184  0.654834
4                         Decision Tree  0.560081   0.444444  0.523560  0.480769  0.592112
5                   Logistic Regression  0.602851   0.484848  0.335079  0.396285  0.584712

### Analysis of Validation Model Comparison

- **Proposed Model - Tuned Random Forest** (`n_estimators=400`, `max_depth=12`, `min_samples_split=4`, `min_samples_leaf=1`, `max_features="sqrt"`) achieves the highest validation **ROC-AUC (`0.6728`)** and highest validation **F1-Score (`0.5356`)** across all six models.
- Compared to the previous median-imputation baseline (`Accuracy=0.6517`, `Precision=0.5714`, `Recall=0.4188`, `F1=0.4834`, `ROC-AUC=0.6594`), the proposed model improves on **every validation metric**.

In [6]:
final_metrics = pd.read_csv("../outputs/final_test_metrics.csv")
final_metrics

                                  Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
0  Proposed Model - Tuned Random Forest  0.676829   0.598802  0.520833  0.557103  0.686337
1                         Random Forest  0.658537   0.573171  0.489583  0.528090  0.689792
2                                   KNN  0.638211   0.561404  0.333333  0.418301  0.615556
3                   Logistic Regression  0.632114   0.536913  0.416667  0.469208  0.631840
4                                   SVM  0.621951   0.515000  0.536458  0.525510  0.658073
5                         Decision Tree  0.589431   0.477679  0.557292  0.514423  0.604071

### Analysis of Final Unseen Test Evaluation and Confusion Matrix

The selected **Proposed Model - Tuned Random Forest** was evaluated on the **492 unseen test samples**, improving every metric and every cell of the confusion matrix over the previous baseline:

- **Accuracy:** **67.68%** *(up from 64.84%, +2.84%)*
- **Precision:** **59.88%** *(up from 55.76%, +4.12%)*
- **Recall:** **52.08%** *(up from 47.92%, +4.16%)*
- **F1-Score:** **55.71%** *(up from 51.54%, +4.17%)*
- **ROC-AUC:** **0.6863** *(up from 0.6758, +0.0105)*
- **Final Confusion Matrix:** `[[233, 67], [92, 100]]` *(up from `[[227, 73], [100, 92]]` — +6 True Negatives, -6 False Positives, -8 False Negatives, +8 True Positives)*

## Conclusion

By combining leak-free multivariate tree-based imputation (`IterativeImputer` with `ExtraTreesRegressor`), domain-driven water chemistry feature engineering (`ph / Sulfate`), and 5-fold Stratified `GridSearchCV` hyperparameter optimization, the proposed **Hyperparameter-Tuned Random Forest** delivers consistent, explainable improvements across validation, 5-fold cross-validation, unseen test metrics, and the final confusion matrix.